In [ ]:
!pip install ampligraph tensorflow keras keras-self-attention

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Curriculum-SessRec-main.zip to Curriculum-SessRec-main.zip


In [ ]:
!unzip "/content/Curriculum-SessRec-main.zip"

Archive:  /content/Curriculum-SessRec-main.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/Curriculum-SessRec-main.zip or
        /content/Curriculum-SessRec-main.zip.zip, and cannot find /content/Curriculum-SessRec-main.zip.ZIP, period.


In [ ]:
import os
os.chdir('/content/Curriculum-SessRec-main')

## 📦 Library Imports

In [ ]:
import numpy as np
import _pickle as cPickle
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Lambda, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from keras_self_attention import SeqSelfAttention
from tqdm import tqdm
import gc

## 2. DATA LOADING & PREP (Self-Contained)

In [ ]:



# ==========================================
# 2. DATA LOADING & PREP (Self-Contained)
# ==========================================
print("🔄 Loading data and initializing vocab...")
with open('Datasets/yoochoose1_64/train.txt', 'rb') as f:
    train = cPickle.load(f)
with open('Datasets/yoochoose1_64/test.txt', 'rb') as f:
    test = cPickle.load(f)

vocab = np.load("Datasets/yoochoose1_64/vocab_yoochoose_64.npy")
total_vocab = len(vocab) + 1
convert_dict = {str(each): i + 1 for i, each in enumerate(vocab)}
convert_dict["padding_id"] = 0

def process_data_safe(data_seq, labels_raw):
    x_out, y_out = [], []
    for seq, label in zip(data_seq, labels_raw):
        processed = [convert_dict.get(str(i), 0) for i in seq]
        if len(processed) > 10: processed = processed[:10]
        else: processed = processed + [0] * (10 - len(processed))
        x_out.append(processed)
        y_out.append(convert_dict.get(str(label), 0))
    return np.array(x_out), np.array(y_out)

train_x, train_y = process_data_safe(train[0], train[1])
test_x, test_y = process_data_safe(test[0], test[1])

# ==========================================
# 3. NOVELTY 3: ADAPTIVE PARAMETERS & ARCHITECTURE
# ==========================================
# Learnable log-variance for weighting
log_var_rec = tf.Variable(0.0, trainable=True, name="log_var_rec", dtype=tf.float32)
log_var_cl = tf.Variable(0.0, trainable=True, name="log_var_cl", dtype=tf.float32)

main_input = Input(shape=(10,))
emb = Embedding(total_vocab, 200, mask_zero=False)(main_input)
lstm_out = LSTM(100, return_sequences=True)(emb)
drop_out = Dropout(0.2)(lstm_out)
attn = SeqSelfAttention(attention_type=SeqSelfAttention.ATTENTION_TYPE_MUL)(drop_out)

# Session latent representation
z = Lambda(lambda xin: K.mean(xin, axis=1), name="latent")(attn)
output = Dense(total_vocab, activation='softmax')(z)

adaptive_model = Model(main_input, [output, z])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# ==========================================
# 4. ADAPTIVE TRAIN STEP
# ==========================================
@tf.function
def adaptive_train_step(xb, yb):
    with tf.GradientTape() as tape:
        preds, z_i = adaptive_model(xb, training=True)
        _, z_j = adaptive_model(xb, training=True)

        # Recommendation Loss
        rec_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(yb, preds))

        # Contrastive Loss (C2)
        z_i_n, z_j_n = tf.math.l2_normalize(z_i, axis=1), tf.math.l2_normalize(z_j, axis=1)
        logits = tf.matmul(z_i_n, z_j_n, transpose_b=True) / 0.07
        cl_loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=tf.range(tf.shape(xb)[0]), logits=logits))

        # Uncertainty Loss Weighting
        precision_rec = tf.exp(-log_var_rec)
        precision_cl = tf.exp(-log_var_cl)

        loss = precision_rec * rec_loss + log_var_rec + precision_cl * cl_loss + log_var_cl

    vars_to_train = adaptive_model.trainable_variables + [log_var_rec, log_var_cl]
    grads = tape.gradient(loss, vars_to_train)
    optimizer.apply_gradients(zip(grads, vars_to_train))
    return loss, rec_loss, cl_loss, precision_rec, precision_cl

# ==========================================
# 5. SAFE EXECUTION LOOP (RAM OPTIMIZED)
# ==========================================
STEPS_PER_EPOCH = 250  # Lowered to prevent crash
BATCH_SIZE = 128       # Lowered for VRAM stability
EPOCHS = 30

print(f"\n🚀 STARTING ADAPTIVE EXPERIMENT | Steps: {STEPS_PER_EPOCH} | Batch: {BATCH_SIZE}")

for epoch in range(EPOCHS):
    indices = np.arange(len(train_x))
    np.random.shuffle(indices)

    pbar = tqdm(total=STEPS_PER_EPOCH, desc=f"Epoch {epoch+1}")
    for s in range(STEPS_PER_EPOCH):
        idx = indices[s*BATCH_SIZE : (s+1)*BATCH_SIZE]
        if len(idx) < BATCH_SIZE: continue

        t_l, r_l, c_l, w_rec, w_cl = adaptive_train_step(train_x[idx], train_y[idx])

        if s % 10 == 0:
            pbar.set_postfix({'W_Rec': f"{w_rec:.2f}", 'W_CL': f"{w_cl:.2f}", 'Rec_L': f"{r_l:.2f}"})
        pbar.update(1)

        # Periodic cleanup every 100 steps to prevent RAM bloat
        if s % 100 == 0:
            gc.collect()

    pbar.close()

    # Streaming Eval (Small slice for speed/safety)
    hits, mrr = 0, 0.0
    val_slice = 1000
    preds_eval, _ = adaptive_model.predict(test_x[:val_slice], batch_size=BATCH_SIZE, verbose=0)

    for i in range(len(preds_eval)):
        top_20 = np.argsort(preds_eval[i])[-20:]
        if test_y[i] in top_20:
            hits += 1
            rank = np.where(top_20[::-1] == test_y[i])[0][0] + 1
            mrr += 1.0 / rank

    print(f"✅ Epoch {epoch+1} -> Recall@20: {(hits/val_slice)*100:.2f}% | MRR@20: {(mrr/val_slice)*100:.2f}%")

    # End of epoch full cleanup
    K.clear_session()
    gc.collect()

print("\nNovelty 3 (Task-Adaptive Weighting) completed successfully.")

🔄 Loading data and initializing vocab...

🚀 STARTING ADAPTIVE EXPERIMENT | Steps: 250 | Batch: 128


Epoch 1: 100%|██████████| 250/250 [00:48<00:00,  5.17it/s, W_Rec=0.80, W_CL=1.26, Rec_L=9.57]


✅ Epoch 1 -> Recall@20: 17.40% | MRR@20: 6.08%


Epoch 2: 100%|██████████| 250/250 [00:44<00:00,  5.61it/s, W_Rec=0.65, W_CL=1.62, Rec_L=9.41]


✅ Epoch 2 -> Recall@20: 19.40% | MRR@20: 6.84%


...

✅ Epoch 28 -> Recall@20: 57.60% | MRR@20: 21.09%


Epoch 29: 100%|██████████| 250/250 [00:46<00:00,  5.43it/s, W_Rec=0.20, W_CL=37.00, Rec_L=5.06]


✅ Epoch 29 -> Recall@20: 58.10% | MRR@20: 21.79%


Epoch 30: 100%|██████████| 250/250 [00:45<00:00,  5.46it/s, W_Rec=0.20, W_CL=37.11, Rec_L=5.37]


✅ Epoch 30 -> Recall@20: 57.40% | MRR@20: 23.08%

Novelty 3 (Task-Adaptive Weighting) completed successfully.
